# 46. MongoDB with PyMongo — `pymongo`
**Python Libraries Lab · Phase 3 · Databases**

_Store flexible JSON-like documents in a NoSQL database._

⚙️ **Runs in Colab** after the setup cell below.

MongoDB is a NoSQL database. Instead of tables with fixed columns, it stores documents, which look like Python dictionaries or JSON. Two products can have different fields, and a document can hold lists and nested objects, like an order with its items inside. PyMongo is the official Python driver.

**In this notebook:** You will store the ShopEasy catalogue and orders as documents, query with filters, update stock, push a review into a product, build a revenue report with an aggregation pipeline, and add an index. The notebook uses a practice in memory database, and connects to a real one if you add a connection string.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
MongoDB is a server. To practise without one, this notebook uses **mongomock**, which copies PyMongo's API in memory. To use a real database, create a free cluster at mongodb.com/atlas, add its connection string to Colab Secrets as `MONGO_URI`, and step 1 will connect to it instead.

The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q pymongo mongomock
import importlib.metadata as md
for pkg in ['pymongo', 'mongomock']:
    print(pkg, md.version(pkg))

## Step 1 · Connect
Uses your real cluster if `MONGO_URI` is set, otherwise the in-memory practice database.

In [ ]:
import os
MONGO_URI = os.environ.get("MONGO_URI")
try:
    from google.colab import userdata
    MONGO_URI = userdata.get("MONGO_URI") or MONGO_URI
except Exception:
    pass

if MONGO_URI:
    import pymongo
    client = pymongo.MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
    print("Connected to your MongoDB server")
else:
    import mongomock
    client = mongomock.MongoClient()
    print("Using the in-memory practice database (mongomock)")
db = client["shopeasy"]
db.products.drop(); db.orders.drop()

## Step 2 · Insert documents
Documents can differ: only some products have `specs` or `tags`.

In [ ]:
db.products.insert_many([
    {"_id": 1, "name": "Steel Bottle", "price": 499, "stock": 40, "tags": ["kitchen", "eco"]},
    {"_id": 2, "name": "Yoga Mat", "price": 1299, "stock": 12, "tags": ["fitness"]},
    {"_id": 3, "name": "Headphones", "price": 2499, "stock": 5, "specs": {"battery_hours": 30, "wireless": True}},
    {"_id": 4, "name": "Notebook", "price": 149, "stock": 200},
])
db.orders.insert_many([
    {"customer": "Asha", "city": "Pune", "items": [{"product_id": 1, "qty": 2}, {"product_id": 4, "qty": 3}], "total": 1445},
    {"customer": "Ravi", "city": "Delhi", "items": [{"product_id": 3, "qty": 1}], "total": 2499},
    {"customer": "Meera", "city": "Pune", "items": [{"product_id": 2, "qty": 1}], "total": 1299},
])
print(db.products.count_documents({}), "products,", db.orders.count_documents({}), "orders")

## Step 3 · Find with filters
`$lt`, `$in`, nested fields with dots, and a projection to choose fields.

In [ ]:
for p in db.products.find({"price": {"$lt": 1000}}, {"name": 1, "price": 1, "_id": 0}).sort("price"):
    print(p)
print(db.products.find_one({"tags": "fitness"})["name"])
print(db.products.find_one({"specs.wireless": True})["name"])
print([o["customer"] for o in db.orders.find({"items.product_id": 1})])

## Step 4 · Update operators
`$inc` changes a number, `$push` adds to a list, `upsert` inserts if missing.

In [ ]:
db.products.update_one({"_id": 3}, {"$inc": {"stock": -1}})
db.products.update_one({"_id": 2}, {"$push": {"reviews": {"user": "Asha", "stars": 4}}})
db.products.update_many({"price": {"$gt": 1000}}, {"$set": {"premium": True}})
db.products.update_one({"_id": 5}, {"$set": {"name": "Desk Lamp", "price": 899, "stock": 7}}, upsert=True)
for p in db.products.find({}, {"_id": 0, "name": 1, "stock": 1, "premium": 1, "reviews": 1}):
    print(p)

## Step 5 · Aggregation pipeline
Stages run in order: group orders by city, then sort by revenue.

In [ ]:
pipeline = [
    {"$group": {"_id": "$city", "orders": {"$sum": 1}, "revenue": {"$sum": "$total"}}},
    {"$sort": {"revenue": -1}},
]
for row in db.orders.aggregate(pipeline):
    print(row)

## Step 6 · Indexes
An index makes lookups on a field fast; `unique` also prevents duplicates.

In [ ]:
db.products.create_index("name", unique=True)
try:
    db.products.insert_one({"name": "Yoga Mat", "price": 999})
except Exception as e:
    print("Blocked duplicate:", type(e).__name__)
print(list(db.products.index_information()))

## ✍️ Your turn
Count the products with **stock below 20** using `count_documents` and store the number in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    return answer == 3, f"Expected 3, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
answer = db.products.count_documents({"stock": {"$lt": 20}})

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **Document**
- **Collection**
- **find / filter**
- **Update operators**
- **Aggregation pipeline**
- **Index**

**Next:** 47. Redis (`redis`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/47_redis.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install pymongo mongomock
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.